In [1]:
from datasets import load_dataset

ds = load_dataset("axmeu/wiki_fr")
train_texts = ds["train"]["text"]
test_texts = ds["test"]["text"]
print(len(test_texts))

/home/onyxia/work/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5000


In [ ]:
train_words = sum(len(text.split()) for text in train_texts)
test_words = sum(len(text.split()) for text in train_texts)

print(f"Total words in train set: {train_words}")
print(f"Total words in test set: {test_words}")


Nombre total de mots dans le train : 317994058


In [2]:
import sys
sys.path.append("..")
from src.BPE.naive import BPE
from src.BPE.fast import FastBPE
from src.WordPiece.naive import WordPiece
from src.WordPiece.fast import FastWordPiece
import time

bpe_naive = BPE.load("../results/models/bpe_fast_v32000_n180000.json")
bpe_fast  = FastBPE.load("../results/models/bpe_fast_v32000_n180000.json")
wp_naive = WordPiece.load("../results/models/wp_fast_v32000_n180000.json")
wp_fast = FastWordPiece.load("../results/models/wp_fast_v32000_n180000.json")

In [13]:
print("BPE:")
s = time.time()
encoded_naive = [bpe_naive.encode(text) for text in test_texts]
print(f"naive encoding in {time.time() - s}")

s = time.time()
encoded_fast = [bpe_fast.encode(text)  for text in test_texts]
print(f"fast encoding in {time.time() - s}")

print("WP:")
s = time.time()
encoded_naive = [wp_naive.encode(text) for text in test_texts]
print(f"naive encoding in {time.time() - s}")

s = time.time()
encoded_fast = [wp_fast.encode(text)  for text in test_texts]
print(f"fast encoding in {time.time() - s}")

BPE:
naive encoding in 5.402616739273071
fast encoding in 0.01073145866394043
WP:
naive encoding in 0.011619806289672852
fast encoding in 0.016795635223388672


In [11]:
from src.WordPiece.naive import WordPiece

wp = WordPiece.load("../results/models/wp_fast_v20000_n180000.json")
print(f"Vocab size: {len(wp.vocab)}")
print(f"Merge rules: {len(wp.merge_rules)}")
print(list(wp.vocab)[:10000])


Vocab size: 20000
Merge rules: 19764
['narration', 'désobéis', 'espace', 'antipersonnel', '##lij', '##Mor', 'Añ', 'réorganisant', 'Holstei', '##rchèren', '##œd', 'gardi', 'Manuels', '##acked', 'Bruche', '##inckelman', 'hercyn', 'Vols', '##rmondial', '##oleplay', '##lahti', '##nebacker', '##issas', 'Toxic', 'infranchissables', '##céra', 'affirmations', '##upai', '##ychothérap', 'revendications', '##zige', '##iéth', 'Z', 'Trygg', '##rancis', '##essuy', '##ostalgiqu', '##istallographique', 'Loxo', '##vrátilo', '##erçants', '##eOffice', 'Equip', 'Pannoni', 'Zacha', '##allonnes', 'über', '##runt', '##nouve', 'distinguai', 'réprimé', '##ezig', '##blayé', 'Brush', 'internements', 'Chiffon', '##obain', 'almohad', 'stal', '##uffets', '##iraux', '##arallèl', '##lifornium', '##ndahl', '##Marc', 'extrayait', 'françai', '##olèse', '##ô', 'jouvence', 'Heels', 'colonisateurs', '##français', 'phrygien', '##ierar', 'Vlaa', 'oxygéné', '##réguennec', 'kw', 'égyptienn', 'dilapid', '##ermondialism', 'envah

In [15]:
seq = "il était est politique époque"
print(bpe_naive.encode(seq))
print(wp_naive.encode(seq))

['i', 'l</w>', 'était</w>', 'est</w>', 'politique</w>', 'époque</w>']
['i', '##l', 'é', '##tait', 'est', 'p', '##o', '##l', '##i', '##t', '##i', '##que', 'é', '##p', '##oque']


In [ ]:
from collections import Counter
import regex

test_word = "con" # vs bon

word_counts = Counter()
for text in train_texts:
    for word in regex.findall(r"[\p{L}\p{N}]+", text):
        word_counts[word] += 1

print(word_counts[test_word])
print(bpe_naive.encode(test_word))

20
['con</w>']


In [7]:
def trace_word(word: str, merge_rules: list) -> None:
    tokens = list(word) + ["</w>"]
    print(f"Initial: {tokens}")
    
    rules_index = {pair: i for i, pair in enumerate(merge_rules)}
    
    step = 0
    while len(tokens) > 1:
        best_idx = None
        best_pos = None
        for i in range(len(tokens) - 1):
            rank = rules_index.get((tokens[i], tokens[i + 1]))
            if rank is not None and (best_idx is None or rank < best_idx):
                best_idx = rank
                best_pos = i
        
        if best_pos is None:
            break
        
        merged = tokens[best_pos] + tokens[best_pos + 1]
        
        print(f"Step {step+1:<4} "
              f"(rule {best_idx:<4}): "
              f"{tokens[best_pos]:<10} + "
              f"{tokens[best_pos+1]:<10} => "
              f"{merged:<20}"
            )

        tokens = tokens[:best_pos] + [merged] + tokens[best_pos + 2:]
        step += 1
    
    print(f"Final: {tokens}")

In [9]:
trace_word("faucon", bpe_fast.merge_rules) # vs charbon

Initial: ['f', 'a', 'u', 'c', 'o', 'n', '</w>']
Step 1    (rule 5   ): o          + n          => on                  
Step 2    (rule 19  ): on         + </w>       => on</w>              
Step 3    (rule 21  ): a          + u          => au                  
Step 4    (rule 3885): c          + on</w>     => con</w>             
Step 5    (rule 15769): au         + con</w>    => aucon</w>           
Step 6    (rule 22108): f          + aucon</w>  => faucon</w>          
Final: ['faucon</w>']
